In [1]:
import pandas as pd

student_info = pd.read_csv("studentInfo.csv")
student_assessment = pd.read_csv("studentAssessment.csv")
assessments = pd.read_csv("assessments.csv")

print("student_info:", student_info.shape)
print("student_assessment:", student_assessment.shape)
print("assessments:", assessments.shape)

display(student_assessment.head())
display(assessments.head())

student_info: (32593, 12)
student_assessment: (173912, 5)
assessments: (206, 6)


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


In [2]:
print("studentAssessment columns:")
print(student_assessment.columns)

print("\nassessments columns:")
print(assessments.columns)

studentAssessment columns:
Index(['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score'], dtype='object')

assessments columns:
Index(['code_module', 'code_presentation', 'id_assessment', 'assessment_type',
       'date', 'weight'],
      dtype='object')


In [3]:
assessment_merged = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)

print("Merged shape:", assessment_merged.shape)
display(assessment_merged.head())

Merged shape: (173912, 10)


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [4]:
print(assessment_merged.isnull().sum())

id_assessment           0
id_student              0
date_submitted          0
is_banked               0
score                 173
code_module             0
code_presentation       0
assessment_type         0
date                 2865
weight                  0
dtype: int64


In [5]:
assessment_merged = assessment_merged.dropna(subset=["score"])

In [6]:
print(assessment_merged.isnull().sum())

id_assessment           0
id_student              0
date_submitted          0
is_banked               0
score                   0
code_module             0
code_presentation       0
assessment_type         0
date                 2865
weight                  0
dtype: int64


In [7]:
print(
    assessment_merged[assessment_merged["date"].isna()]
    ["assessment_type"]
    .value_counts()
)

assessment_type
Exam    2865
Name: count, dtype: int64


In [8]:
assessment_merged_no_exam = assessment_merged[
    assessment_merged["assessment_type"] != "Exam"
].copy()

print("Before removing Exam:", assessment_merged.shape)
print("After removing Exam:", assessment_merged_no_exam.shape)

print(assessment_merged_no_exam["assessment_type"].value_counts())
print(assessment_merged_no_exam["date"].isnull().sum())

Before removing Exam: (173739, 10)
After removing Exam: (168780, 10)
assessment_type
TMA    98253
CMA    70527
Name: count, dtype: int64
0


In [9]:
assessment_merged_no_exam["submitted_late"] = (
    assessment_merged_no_exam["date_submitted"] > assessment_merged_no_exam["date"]
).astype(int)

assessment_merged_no_exam["submission_delay"] = (
    assessment_merged_no_exam["date_submitted"] - assessment_merged_no_exam["date"]
)

assessment_merged_no_exam["weighted_score"] = (
    assessment_merged_no_exam["score"] * assessment_merged_no_exam["weight"]
)

In [10]:
import numpy as np

assessment_features = assessment_merged_no_exam.groupby(
    ["id_student", "code_module", "code_presentation"]
).agg(
    avg_score=("score", "mean"),
    max_score=("score", "max"),
    min_score=("score", "min"),
    score_std=("score", "std"),
    submitted_assessments_count=("id_assessment", "count"),
    late_submissions_count=("submitted_late", "sum"),
    avg_submission_delay=("submission_delay", "mean"),
    total_weight_attempted=("weight", "sum"),
    total_weighted_score=("weighted_score", "sum")
).reset_index()

assessment_features["weighted_score_avg"] = (
    assessment_features["total_weighted_score"] /
    assessment_features["total_weight_attempted"]
)

assessment_features["late_submission_rate"] = (
    assessment_features["late_submissions_count"] /
    assessment_features["submitted_assessments_count"]
)

assessment_features["weighted_score_avg"] = assessment_features["weighted_score_avg"].replace(
    [np.inf, -np.inf],
    np.nan
)

assessment_features["weighted_score_avg"] = assessment_features["weighted_score_avg"].fillna(
    assessment_features["avg_score"]
)

assessment_features["score_std"] = assessment_features["score_std"].fillna(0)

print("assessment_features shape:", assessment_features.shape)
display(assessment_features.head())
print(assessment_features.isnull().sum())

assessment_features shape: (25816, 14)


,id_student,code_module,code_presentation,avg_score,max_score,min_score,score_std,submitted_assessments_count,late_submissions_count,avg_submission_delay,total_weight_attempted,total_weighted_score,weighted_score_avg,late_submission_rate
0,6516,AAA,2014J,61.800000,77.0,48.0,10.329569,5,0,-2.600000,100.0,6350.0,63.50,0.000000
1,8462,DDD,2013J,87.666667,93.0,83.0,5.033223,3,1,-0.333333,40.0,3490.0,87.25,0.333333
2,8462,DDD,2014J,86.500000,93.0,83.0,4.725816,4,0,-59.500000,50.0,4300.0,86.00,0.000000
3,11391,AAA,2013J,82.000000,85.0,78.0,3.082207,5,0,-1.800000,100.0,8240.0,82.40,0.000000
4,23629,BBB,2013B,82.500000,100.0,63.0,20.273135,4,3,3.500000,25.0,1669.0,66.76,0.750000


id_student                     0
code_module                    0
code_presentation              0
avg_score                      0
max_score                      0
min_score                      0
score_std                      0
submitted_assessments_count    0
late_submissions_count         0
avg_submission_delay           0
total_weight_attempted         0
total_weighted_score           0
weighted_score_avg             0
late_submission_rate           0
dtype: int64


In [11]:
data_v2 = student_info.copy()

# Handle missing values in imd_band
data_v2["imd_band"] = data_v2["imd_band"].fillna("Unknown")

# Create binary target
data_v2["at_risk"] = data_v2["final_result"].apply(
    lambda x: 1 if x in ["Fail", "Withdrawn"] else 0
)

# Merge assessment features
data_v2 = data_v2.merge(
    assessment_features,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

print("data_v2 shape:", data_v2.shape)
display(data_v2.head())
print(data_v2.isnull().sum())

data_v2 shape: (32593, 24)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,max_score,min_score,score_std,submitted_assessments_count,late_submissions_count,avg_submission_delay,total_weight_attempted,total_weighted_score,weighted_score_avg,late_submission_rate
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,85.0,78.0,3.082207,5.0,0.0,-1.8,100.0,8240.0,82.4,0.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,70.0,60.0,4.335897,5.0,2.0,0.0,100.0,6540.0,65.4,0.4
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,88.0,71.0,6.892024,5.0,0.0,-2.0,100.0,7630.0,76.3,0.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,75.0,30.0,20.513410,5.0,5.0,11.4,100.0,5500.0,55.0,1.0


code_module                       0
code_presentation                 0
id_student                        0
gender                            0
region                            0
highest_education                 0
imd_band                          0
age_band                          0
num_of_prev_attempts              0
studied_credits                   0
disability                        0
final_result                      0
at_risk                           0
avg_score                      6777
max_score                      6777
min_score                      6777
score_std                      6777
submitted_assessments_count    6777
late_submissions_count         6777
avg_submission_delay           6777
total_weight_attempted         6777
total_weighted_score           6777
weighted_score_avg             6777
late_submission_rate           6777
dtype: int64


In [12]:
assessment_numeric_features = [
    "avg_score",
    "max_score",
    "min_score",
    "score_std",
    "submitted_assessments_count",
    "late_submissions_count",
    "avg_submission_delay",
    "total_weight_attempted",
    "total_weighted_score",
    "weighted_score_avg",
    "late_submission_rate"
]

data_v2[assessment_numeric_features] = data_v2[assessment_numeric_features].fillna(0)

In [13]:
print(data_v2.isnull().sum())

code_module                    0
code_presentation              0
id_student                     0
gender                         0
region                         0
highest_education              0
imd_band                       0
age_band                       0
num_of_prev_attempts           0
studied_credits                0
disability                     0
final_result                   0
at_risk                        0
avg_score                      0
max_score                      0
min_score                      0
score_std                      0
submitted_assessments_count    0
late_submissions_count         0
avg_submission_delay           0
total_weight_attempted         0
total_weighted_score           0
weighted_score_avg             0
late_submission_rate           0
dtype: int64


In [14]:
base_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability"
]

features_v2 = base_features + assessment_numeric_features

X = data_v2[features_v2]
y = data_v2["at_risk"]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

X shape: (32593, 21)
y shape: (32593,)


,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,...,max_score,min_score,score_std,submitted_assessments_count,late_submissions_count,avg_submission_delay,total_weight_attempted,total_weighted_score,weighted_score_avg,late_submission_rate
0,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,...,85.0,78.0,3.082207,5.0,0.0,-1.8,100.0,8240.0,82.4,0.0
1,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,...,70.0,60.0,4.335897,5.0,2.0,0.0,100.0,6540.0,65.4,0.4
2,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,...,88.0,71.0,6.892024,5.0,0.0,-2.0,100.0,7630.0,76.3,0.0
4,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,...,75.0,30.0,20.513410,5.0,5.0,11.4,100.0,5500.0,55.0,1.0


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)

Train shape: (26074, 21)
Test shape: (6519, 21)
at_risk
1    52.795889
0    47.204111
Name: proportion, dtype: float64
at_risk
1    52.799509
0    47.200491
Name: proportion, dtype: float64


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits"
] + assessment_numeric_features

preprocessor_v2 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

xgb_model_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]
)

xgb_model_v2.fit(X_train, y_train)

y_pred_xgb_v2 = xgb_model_v2.predict(X_test)

print("XGBoost V2 Accuracy:", accuracy_score(y_test, y_pred_xgb_v2))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_v2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_v2))

XGBoost V2 Accuracy: 0.926829268292683

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.96      0.93      3077
           1       0.96      0.90      0.93      3442

    accuracy                           0.93      6519
   macro avg       0.93      0.93      0.93      6519
weighted avg       0.93      0.93      0.93      6519


Confusion Matrix:
[[2952  125]
 [ 352 3090]]


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf_model_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

rf_model_v2.fit(X_train, y_train)

y_pred_rf_v2 = rf_model_v2.predict(X_test)

print("Random Forest V2 Accuracy:", accuracy_score(y_test, y_pred_rf_v2))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_v2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_v2))

Random Forest V2 Accuracy: 0.9279030526154318

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.93      3077
           1       0.96      0.90      0.93      3442

    accuracy                           0.93      6519
   macro avg       0.93      0.93      0.93      6519
weighted avg       0.93      0.93      0.93      6519


Confusion Matrix:
[[2934  143]
 [ 327 3115]]


-----------

In [19]:
early_day_cutoff = 120

assessment_early = assessment_merged_no_exam[
    assessment_merged_no_exam["date"] <= early_day_cutoff
].copy()

print("Full no-exam assessments:", assessment_merged_no_exam.shape)
print("Early assessments:", assessment_early.shape)

print(assessment_early["assessment_type"].value_counts())
print("Max assessment date:", assessment_early["date"].max())
print("Min assessment date:", assessment_early["date"].min())

Full no-exam assessments: (168780, 13)
Early assessments: (79936, 13)
assessment_type
TMA    62606
CMA    17330
Name: count, dtype: int64
Max assessment date: 117.0
Min assessment date: 12.0


In [20]:
import numpy as np

assessment_early["submitted_late"] = (
    assessment_early["date_submitted"] > assessment_early["date"]
).astype(int)

assessment_early["submission_delay"] = (
    assessment_early["date_submitted"] - assessment_early["date"]
)

assessment_early["weighted_score"] = (
    assessment_early["score"] * assessment_early["weight"]
)

assessment_features_early = assessment_early.groupby(
    ["id_student", "code_module", "code_presentation"]
).agg(
    early_avg_score=("score", "mean"),
    early_max_score=("score", "max"),
    early_min_score=("score", "min"),
    early_score_std=("score", "std"),
    early_submitted_assessments_count=("id_assessment", "count"),
    early_late_submissions_count=("submitted_late", "sum"),
    early_avg_submission_delay=("submission_delay", "mean"),
    early_total_weight_attempted=("weight", "sum"),
    early_total_weighted_score=("weighted_score", "sum")
).reset_index()

assessment_features_early["early_weighted_score_avg"] = (
    assessment_features_early["early_total_weighted_score"] /
    assessment_features_early["early_total_weight_attempted"]
)

assessment_features_early["early_late_submission_rate"] = (
    assessment_features_early["early_late_submissions_count"] /
    assessment_features_early["early_submitted_assessments_count"]
)

assessment_features_early["early_weighted_score_avg"] = assessment_features_early["early_weighted_score_avg"].replace(
    [np.inf, -np.inf],
    np.nan
)

assessment_features_early["early_weighted_score_avg"] = assessment_features_early["early_weighted_score_avg"].fillna(
    assessment_features_early["early_avg_score"]
)

assessment_features_early["early_score_std"] = assessment_features_early["early_score_std"].fillna(0)

print("assessment_features_early shape:", assessment_features_early.shape)
display(assessment_features_early.head())
print(assessment_features_early.isnull().sum())

assessment_features_early shape: (25724, 14)


,id_student,code_module,code_presentation,early_avg_score,early_max_score,early_min_score,early_score_std,early_submitted_assessments_count,early_late_submissions_count,early_avg_submission_delay,early_total_weight_attempted,early_total_weighted_score,early_weighted_score_avg,early_late_submission_rate
0,6516,AAA,2014J,57.000000,63.0,48.0,7.937254,3,0,-2.000000,50.0,2820.0,56.40,0.000000
1,8462,DDD,2013J,87.666667,93.0,83.0,5.033223,3,1,-0.333333,40.0,3490.0,87.25,0.333333
2,8462,DDD,2014J,86.500000,93.0,83.0,4.725816,4,0,-59.500000,50.0,4300.0,86.00,0.000000
3,11391,AAA,2013J,81.000000,85.0,78.0,3.605551,3,0,-1.333333,50.0,4080.0,81.60,0.000000
4,23629,BBB,2013B,82.500000,100.0,63.0,20.273135,4,3,3.500000,25.0,1669.0,66.76,0.750000


id_student                           0
code_module                          0
code_presentation                    0
early_avg_score                      0
early_max_score                      0
early_min_score                      0
early_score_std                      0
early_submitted_assessments_count    0
early_late_submissions_count         0
early_avg_submission_delay           0
early_total_weight_attempted         0
early_total_weighted_score           0
early_weighted_score_avg             0
early_late_submission_rate           0
dtype: int64


In [21]:
data_v2_early = student_info.copy()

data_v2_early["imd_band"] = data_v2_early["imd_band"].fillna("Unknown")

data_v2_early["at_risk"] = data_v2_early["final_result"].apply(
    lambda x: 1 if x in ["Fail", "Withdrawn"] else 0
)

data_v2_early = data_v2_early.merge(
    assessment_features_early,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

early_assessment_numeric_features = [
    "early_avg_score",
    "early_max_score",
    "early_min_score",
    "early_score_std",
    "early_submitted_assessments_count",
    "early_late_submissions_count",
    "early_avg_submission_delay",
    "early_total_weight_attempted",
    "early_total_weighted_score",
    "early_weighted_score_avg",
    "early_late_submission_rate"
]

data_v2_early[early_assessment_numeric_features] = data_v2_early[early_assessment_numeric_features].fillna(0)

print("data_v2_early shape:", data_v2_early.shape)
print(data_v2_early[early_assessment_numeric_features].isnull().sum())

data_v2_early shape: (32593, 24)
early_avg_score                      0
early_max_score                      0
early_min_score                      0
early_score_std                      0
early_submitted_assessments_count    0
early_late_submissions_count         0
early_avg_submission_delay           0
early_total_weight_attempted         0
early_total_weighted_score           0
early_weighted_score_avg             0
early_late_submission_rate           0
dtype: int64


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

base_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability"
]

features_v2_early = base_features + early_assessment_numeric_features

X = data_v2_early[features_v2_early]
y = data_v2_early["at_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits"
] + early_assessment_numeric_features

preprocessor_v2_early = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

xgb_model_v2_early = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2_early),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]
)

xgb_model_v2_early.fit(X_train, y_train)

y_pred_xgb_early = xgb_model_v2_early.predict(X_test)

print("XGBoost V2 Early Accuracy:", accuracy_score(y_test, y_pred_xgb_early))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_early))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_early))

XGBoost V2 Early Accuracy: 0.8694585059058137

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.92      0.87      3077
           1       0.92      0.83      0.87      3442

    accuracy                           0.87      6519
   macro avg       0.87      0.87      0.87      6519
weighted avg       0.87      0.87      0.87      6519


Confusion Matrix:
[[2823  254]
 [ 597 2845]]


The early assessment-based model used only assessment records within the first 60 days of the course and excluded final exams to reduce temporal data leakage. Compared with the demographic baseline, the model improved accuracy from 63.46% to 80.93%. The At Risk F1-score also increased from 0.66 to 0.81, indicating that early assessment behavior provides a strong predictive signal for identifying students at risk.